# PCAP moving-object tracking app

Upload a LiDAR `.pcap`, decode it into point-cloud frames, set a frame range, and enter an axis-aligned manual bounding box for each frame. The app compares a Kalman filter or particle filter against those manual boxes.

The decoder below targets Velodyne-style UDP packets (including VLP-16/VLP-32 layouts). For another sensor, replace `decode_lidar_packet` while keeping its `(x, y, z)` output contract.

In [ ]:
import io
import math
import struct
import dpkt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import clear_output, display

## PCAP decoding and frame construction

In [ ]:

_16_ANGLES = np.array([-15, 1, -13, 3, -11, 5, -9, 7, -7, 9, -5, 11, -3, 13, -1, 15], dtype=float)
_32_ANGLES = np.array([-30.67, -9.329, -29.333, -8.0, -28.0, -6.667, -26.667, -5.333, -25.333, -4.0, -24.0, -2.667, -22.667, -1.333, -21.333, 0.0, -20.0, 1.333, -18.667, 2.667, -17.333, 4.0, -16.0, 5.333, -14.667, 6.667, -13.333, 8.0, -12.0, 9.333, -10.667, 10.667], dtype=float)

def decode_lidar_packet(payload, sensor_channels=16, distance_scale=0.002):
    angles = _16_ANGLES if sensor_channels == 16 else _32_ANGLES
    points = []
    for block_start in range(0, len(payload) - 99, 100):
        if payload[block_start:block_start + 2] != b'\xff\xee':
            continue
        azimuth = struct.unpack_from('<H', payload, block_start + 2)[0] / 100.0
        azimuth_rad = math.radians(azimuth)
        for channel in range(sensor_channels):
            offset = block_start + 4 + channel * 3
            distance = struct.unpack_from('<H', payload, offset)[0] * distance_scale
            if distance <= 0:
                continue
            vertical_rad = math.radians(angles[channel])
            horizontal = distance * math.cos(vertical_rad)
            points.append((horizontal * math.sin(azimuth_rad), horizontal * math.cos(azimuth_rad), distance * math.sin(vertical_rad)))
    return np.asarray(points, dtype=float).reshape(-1, 3)


def pcap_to_frames(file_bytes, packets_per_frame=20, sensor_channels=16, max_frames=200):
    """Read UDP packets and group decoded packets into point-cloud frames."""
    frames = []
    packet_points = []
    reader = dpkt.pcap.Reader(io.BytesIO(file_bytes))
    for _, raw_packet in reader:
        try:
            ethernet = dpkt.ethernet.Ethernet(raw_packet)
            ip_packet = ethernet.data
            udp_packet = ip_packet.data
            payload = bytes(udp_packet.data)
        except (AttributeError, dpkt.dpkt.NeedData, ValueError):
            continue
        decoded = decode_lidar_packet(payload, sensor_channels=sensor_channels)
        if len(decoded):
            packet_points.append(decoded)
        if len(packet_points) >= packets_per_frame:
            frames.append(np.vstack(packet_points))
            packet_points = []
            if len(frames) >= max_frames:
                break
    if packet_points and len(frames) < max_frames:
        frames.append(np.vstack(packet_points))
    if not frames:
        raise ValueError('No decodable LiDAR points were found in the PCAP.')
    return frames

## Filters and box utilities

In [ ]:
def box_center(box):
    return np.array([(box['xmin'] + box['xmax']) / 2, (box['ymin'] + box['ymax']) / 2, (box['zmin'] + box['zmax']) / 2], dtype=float)


def box_size(box):
    return np.array([box['xmax'] - box['xmin'], box['ymax'] - box['ymin'], box['zmax'] - box['zmin']], dtype=float)


def points_in_box(points, box):
    lower = np.array([box['xmin'], box['ymin'], box['zmin']])
    upper = np.array([box['xmax'], box['ymax'], box['zmax']])
    return points[np.all((points >= lower) & (points <= upper), axis=1)]


class KalmanCenterFilter:
    def __init__(self, initial_center, dt=1.0, process_noise=0.05, measurement_noise=0.2):
        self.state = np.r_[initial_center, np.zeros(3)]
        self.F = np.block([[np.eye(3), dt * np.eye(3)], [np.zeros((3, 3)), np.eye(3)]])
        self.H = np.block([np.eye(3), np.zeros((3, 3))])
        self.P = np.eye(6)
        self.Q = process_noise * np.eye(6)
        self.R = measurement_noise * np.eye(3)

    def predict(self):
        self.state = self.F @ self.state
        self.P = self.F @ self.P @ self.F.T + self.Q
        return self.state[:3].copy()

    def update(self, measurement):
        innovation = measurement - self.H @ self.state
        covariance = self.H @ self.P @ self.H.T + self.R
        gain = self.P @ self.H.T @ np.linalg.inv(covariance)
        self.state += gain @ innovation
        self.P = (np.eye(6) - gain @ self.H) @ self.P
        return self.state[:3].copy()


class ParticleCenterFilter:
    def __init__(self, initial_center, particle_count=300, motion_noise=0.15, seed=42):
        self.rng = np.random.default_rng(seed)
        self.particles = self.rng.normal(initial_center, 0.05, size=(particle_count, 3))
        self.motion_noise = motion_noise

    def predict(self):
        self.particles += self.rng.normal(0, self.motion_noise, self.particles.shape)
        return self.particles.mean(axis=0)

    def update(self, measurement):
        distances = np.linalg.norm(self.particles - measurement, axis=1)
        weights = np.exp(-0.5 * (distances / max(self.motion_noise, 1e-6)) ** 2) + 1e-12
        weights /= weights.sum()
        indices = self.rng.choice(len(self.particles), len(self.particles), p=weights)
        self.particles = self.particles[indices] + self.rng.normal(0, self.motion_noise * 0.25, self.particles.shape)
        return self.particles.mean(axis=0)


def detect_object_center(points, predicted_center, search_radius, fallback):
    distances = np.linalg.norm(points - predicted_center, axis=1)
    candidates = points[distances <= search_radius]
    return candidates.mean(axis=0) if len(candidates) else fallback

## Interactive application

In [ ]:
state = {'frames': [], 'manual_boxes': {}, 'tracking': {}}
upload = widgets.FileUpload(accept='.pcap', multiple=False, description='Open PCAP')
frame_count = widgets.IntSlider(value=50, min=1, max=500, step=1, description='Frames', continuous_update=False)
packets_per_frame = widgets.IntSlider(value=20, min=1, max=200, step=1, description='Packets/frame', continuous_update=False)
sensor_channels = widgets.Dropdown(options=[16, 32], value=16, description='Channels')
frame_index = widgets.IntSlider(value=0, min=0, max=0, description='Manual frame', continuous_update=False)
filter_choice = widgets.ToggleButtons(options=['Kalman', 'Particle'], description='Filter')
search_radius = widgets.FloatText(value=1.0, description='Search radius')
xmin = widgets.FloatText(description='xmin'); xmax = widgets.FloatText(description='xmax')
ymin = widgets.FloatText(description='ymin'); ymax = widgets.FloatText(description='ymax')
zmin = widgets.FloatText(description='zmin'); zmax = widgets.FloatText(description='zmax')
load_button = widgets.Button(description='Decode frames', button_style='primary', icon='upload')
add_box_button = widgets.Button(description='Save box for frame', button_style='success', icon='check')
run_button = widgets.Button(description='Track and compare', button_style='warning', icon='play')
status = widgets.HTML(value='Open a PCAP file to begin.')
box_output = widgets.Output(); result_output = widgets.Output(); plot_output = widgets.Output()

In [ ]:
def set_status(message, color='#444'):
    status.value = f"<span style='color:{color}'>{message}</span>"


def load_uploaded_pcap(_):
    if not upload.value:
        set_status('Choose a PCAP file first.', '#a11')
        return
    uploaded = next(iter(upload.value.values())) if isinstance(upload.value, dict) else upload.value[0]
    try:
        state['frames'] = pcap_to_frames(uploaded['content'], packets_per_frame.value, sensor_channels.value, frame_count.value)
        frame_index.max = max(len(state['frames']) - 1, 0)
        state['manual_boxes'] = {}
        set_status(f"Decoded {len(state['frames'])} frames from {uploaded['name']}.", '#176b3a')
        show_frame()
    except Exception as error:
        state['frames'] = []
        set_status(str(error), '#a11')


def show_frame(_=None):
    if not state['frames']:
        return
    points = state['frames'][frame_index.value]
    if len(points):
        lower, upper = points.min(axis=0), points.max(axis=0)
        for field, value in zip((xmin, ymin, zmin), lower): field.value = float(value)
        for field, value in zip((xmax, ymax, zmax), upper): field.value = float(value)
    saved = state['manual_boxes'].get(frame_index.value)
    if saved:
        for field, key in zip((xmin, xmax, ymin, ymax, zmin, zmax), ('xmin', 'xmax', 'ymin', 'ymax', 'zmin', 'zmax')): field.value = saved[key]
    with box_output:
        clear_output(wait=True)
        display(widgets.HTML(f"Frame {frame_index.value}: {len(points):,} points. Saved boxes: {len(state['manual_boxes'])}"))


def save_manual_box(_):
    if not state['frames']:
        set_status('Decode frames before adding manual boxes.', '#a11')
        return
    box = {key: float(field.value) for key, field in zip(('xmin', 'xmax', 'ymin', 'ymax', 'zmin', 'zmax'), (xmin, xmax, ymin, ymax, zmin, zmax))}
    if any(box[f'{axis}min'] > box[f'{axis}max'] for axis in ('x', 'y', 'z')):
        set_status('Each minimum must be less than or equal to its maximum.', '#a11')
        return
    state['manual_boxes'][frame_index.value] = box
    set_status(f"Saved manual box for frame {frame_index.value}.", '#176b3a')
    show_frame()


upload.observe(lambda change: set_status('PCAP selected. Click Decode frames.', '#176b3a'), names='value')
load_button.on_click(load_uploaded_pcap)
frame_index.observe(show_frame, names='value')
add_box_button.on_click(save_manual_box)

In [ ]:
def iou_3d(first, second):
    first_lower = np.array([first['xmin'], first['ymin'], first['zmin']]); first_upper = np.array([first['xmax'], first['ymax'], first['zmax']])
    second_lower = np.array([second['xmin'], second['ymin'], second['zmin']]); second_upper = np.array([second['xmax'], second['ymax'], second['zmax']])
    intersection_size = np.maximum(0, np.minimum(first_upper, second_upper) - np.maximum(first_lower, second_lower))
    intersection = np.prod(intersection_size)
    volume_first = np.prod(first_upper - first_lower); volume_second = np.prod(second_upper - second_lower)
    return float(intersection / max(volume_first + volume_second - intersection, 1e-12))


def run_tracking(_):
    if not state['frames'] or not state['manual_boxes']:
        set_status('Decode frames and save at least one manual box first.', '#a11')
        return
    start = min(state['manual_boxes'])
    initial_box = state['manual_boxes'][start]
    initial_center = box_center(initial_box)
    filter_object = KalmanCenterFilter(initial_center) if filter_choice.value == 'Kalman' else ParticleCenterFilter(initial_center)
    results = []
    previous_center = initial_center
    reference_size = box_size(initial_box)
    for index in range(start, len(state['frames'])):
        predicted = filter_object.predict() if index > start else initial_center
        measurement = detect_object_center(state['frames'][index], predicted, search_radius.value, previous_center)
        estimated_center = filter_object.update(measurement)
        previous_center = estimated_center
        estimated = {'xmin': estimated_center[0] - reference_size[0] / 2, 'xmax': estimated_center[0] + reference_size[0] / 2, 'ymin': estimated_center[1] - reference_size[1] / 2, 'ymax': estimated_center[1] + reference_size[1] / 2, 'zmin': estimated_center[2] - reference_size[2] / 2, 'zmax': estimated_center[2] + reference_size[2] / 2}
        manual = state['manual_boxes'].get(index)
        results.append({'frame': index, 'center_error': np.nan if manual is None else float(np.linalg.norm(estimated_center - box_center(manual))), 'iou': np.nan if manual is None else iou_3d(estimated, manual), 'estimated': estimated, 'manual': manual})
    state['tracking'] = results
    table = pd.DataFrame([{'frame': row['frame'], 'center_error': row['center_error'], 'iou_3d': row['iou']} for row in results])
    with result_output:
        clear_output(wait=True)
        display(table)
        display(widgets.HTML(f"Mean center error: {table['center_error'].mean():.3f} | Mean 3D IoU: {table['iou_3d'].mean():.3f}"))
    with plot_output:
        clear_output(wait=True)
        figure, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
        axes[0].plot(table['frame'], table['center_error'], marker='o'); axes[0].set_ylabel('Center error'); axes[0].grid(True)
        axes[1].plot(table['frame'], table['iou_3d'], marker='o'); axes[1].set_ylabel('3D IoU'); axes[1].set_xlabel('Frame'); axes[1].set_ylim(0, 1); axes[1].grid(True)
        figure.tight_layout(); plt.show()
    set_status(f"Tracked {len(results)} frames with {filter_choice.value} filter.", '#176b3a')


run_button.on_click(run_tracking)
display(widgets.VBox([upload, widgets.HBox([frame_count, packets_per_frame, sensor_channels]), load_button, status, frame_index, box_output, widgets.HTML('<b>Manual box for selected frame</b>'), widgets.HBox([xmin, xmax]), widgets.HBox([ymin, ymax]), widgets.HBox([zmin, zmax]), add_box_button, widgets.HTML('<b>Tracking</b>'), widgets.HBox([filter_choice, search_radius]), run_button, result_output, plot_output]))